## Patterns with two members

Modifying two-member patterns into a format which allows them to be saved into the same table as one-member patterns. This means that pattern members are splitted into two parts that are saved in separate table rows that share the same pattern ID. A new database will be created with a table of modified patterns. This table will be later used in creating a *patterns* table of all patterns (both length 1 and length 2) in *04_creating_vp_data4.ipynb*.

In [1]:
import sys

sys.path.append('../../../common_code')

In [2]:
import sqlite3
from db_operations.db_display import *

## Input parameters

In [3]:
INPUT_DIR = "../001_creating_pattern_tables"

RESULT_DB = "vp_len2.db"
VERB_PATTERN_DB = f"{INPUT_DIR}/verb_patterns_new.db"

## Data processing

In [4]:
con = sqlite3.connect(VERB_PATTERN_DB)
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{RESULT_DB}" AS vp2')

# patterns_len2 tabelist vajaliku sisu sisselugemine
# võetakse ainult read, kus mustris puudub 'other' kategooriasse kuuluv liige ning verbikonstruktsioon on kuni kaheliikmeline
cur.execute("""
SELECT
    ID,
    word,
    government,
    verb_word,
    compound_prt1,
    member1_case,
    member1_adp,
    member2_case,
    member2_adp,
    member2_verb  
FROM
    patterns_len2
WHERE
    compound_prt2=''
AND
    compound_prt3=''
AND
    member1_other=''
AND
    member2_other=''
""")

rows = cur.fetchall()

In [5]:
# patterns_len1 pikkus
# (vajalik, et indekseerida mustreid pikkusega 2, indeksid peaksid jätkama patterns_len1 indekseid)
cur.execute("""
SELECT * FROM patterns_len1
""")

n_pat_len1 = len(cur.fetchall())

In [6]:
# mustri liikmed eraldatakse ning sisestatakse uude andmebaasi
# esimene ja teine liige sisestatakse eraldi ridadele, kuid neid ühendab identne mustri ID

# indeksid
# 0 - ID
# 1 - word
# 2 - government
# 3 - verb_word
# 4 - compound_prt
# 5 - member1_case
# 6 - member1_adp
# 7 - member2_case
# 8 - member2_adp
# 9 - member2_verb

cur.execute("""
DROP TABLE IF EXISTS vp2.temp_patterns
""")

cur.execute("""
CREATE TABLE vp2.temp_patterns(
    pat_id INTEGER,
    pattern TEXT,
    verb_word TEXT,
    verb_compound TEXT,
    phrase_nr INTEGER,
    phrase_case TEXT,
    adp TEXT,
    inf_verb TEXT
)
""")

pat_idx = n_pat_len1+1
for i in range(len(rows)):
    pattern = rows[i][1]+' '+rows[i][2]
    cur.execute("""
    INSERT INTO vp2.temp_patterns
    (pat_id, pattern, verb_word, verb_compound, phrase_nr, phrase_case, adp, inf_verb)
    VALUES
    (?, ?, ?, ?, ?, ?, ?, ?)
    """, (pat_idx, pattern, rows[i][3], rows[i][4], 1, rows[i][5], rows[i][6], ''))
    cur.execute("""
    INSERT INTO vp2.temp_patterns
    (pat_id, pattern, verb_word, verb_compound, phrase_nr, phrase_case, adp, inf_verb)
    VALUES
    (?, ?, ?, ?, ?, ?, ?, ?)
    """, (pat_idx, pattern, rows[i][3], rows[i][4], 2, rows[i][7], rows[i][8], rows[i][9]))
    pat_idx+=1
    con.commit()

con.close()

## Result

In [7]:
display_sqlite_as_dataframe(RESULT_DB, 'temp_patterns', 10)

,pat_id,pattern,verb_word,verb_compound,phrase_nr,phrase_case,adp,inf_verb
0,2527,abistama keda/mida* + milles,abistama,,1,part,,
1,2527,abistama keda/mida* + milles,abistama,,2,in,,
2,2528,aitama kellel + mida teha,aitama,,1,ad,,
3,2528,aitama kellel + mida teha,aitama,,2,part,,teha
4,2529,alla kirjutama mille/mida,kirjutama,alla,1,part,,
5,2529,alla kirjutama mille/mida,kirjutama,alla,2,gen,,
6,2530,ette heitma kellele/millele + mida*,heitma,ette,1,all,,
7,2530,ette heitma kellele/millele + mida*,heitma,ette,2,part,,
8,2531,informeerima keda* + millest,informeerima,,1,part,,
9,2531,informeerima keda* + millest,informeerima,,2,el,,
